# Part D: Food-order stream with RabbitMQ

Run the cells in order. Complete your D1 routing predictions before the demonstration cell. The cells headed **SCREENSHOT** show exactly what to capture for your report.

## D1 - routing predictions (complete before running)

For each routing key in the assignment, write the queues you predict will receive it and explain the two-word key. This is a written table for your report; it does not require a Colab screenshot.

In [1]:
!git clone https://github.com/AK-CODE-droid/freshcart-pipelines.git
%cd freshcart-pipelines
!git status

Cloning into 'freshcart-pipelines'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 73 (delta 14), reused 68 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 838.70 KiB | 15.53 MiB/s, done.
Resolving deltas: 100% (14/14), done.
/content/freshcart-pipelines
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [2]:
!apt-get update -qq
!apt-get install -y -qq rabbitmq-server
!service rabbitmq-server start
!pip install -q pika
!rabbitmqctl status

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package logrotate.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../00-logrotate_3.21.0-2build1_amd64.deb ...
Unpacking logrotate (3.21.0-2build1) ...
Selecting previously unselected package erlang-base.
Preparing to unpack .../01-erlang-base_1%3a25.3.2.8+dfsg-1ubuntu4.8_amd64.deb ...
Unpacking erlang-base (1:25.3.2.8+dfsg-1ubuntu4.8) ...
Selecting previously unselected package erlang-asn1.
Preparing to unpack .../02-erlang-asn1_1%3a25.3.2.8+dfsg-1ubuntu4.8_amd64.deb ...
Unpacking erlang-asn1 (1:25.3.2.8+dfsg-1ubuntu4.8) ...
Selecting previously unselected package erlang-crypto.
Preparing to unpack .../03-erlang-crypto_1%3a25.3.2.8+dfsg-1ubuntu4.8_amd64.deb ...
Unpacking erlang-crypto (1:25.3.2.8+dfsg-1ubuntu4.8) ...
Sele

## Create `stream/dispatcher.py`

In [3]:
%%writefile stream/dispatcher.py
import argparse
import json
import random
import time
import uuid
from datetime import datetime, timezone

import pika

EXCHANGE_NAME = 'rapido-dispatch'
LOCATIONS = {
    'bangalore': ('koramangala', 'indiranagar', 'whitefield'),
    'chennai': ('tnagar', 'whitefield'),
}
REQUEST_TYPES = ('ride', 'food')


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument('--interval', type=float, default=2.0)
    parser.add_argument('--bad-every', type=int, default=10)
    parser.add_argument('--proof-key')
    return parser.parse_args()


def publish(channel, routing_key, request, label=''):
    channel.basic_publish(
        exchange=EXCHANGE_NAME,
        routing_key=routing_key,
        body=json.dumps(request),
        properties=pika.BasicProperties(content_type='application/json'),
    )
    print(f'{label}published {routing_key}: {request}')


def make_request(city, zone, request_type):
    return {
        'id': str(uuid.uuid4()),
        'city': city,
        'zone': zone,
        'type': request_type,
        'ts': datetime.now(timezone.utc).isoformat(),
    }


def main():
    args = parse_args()
    connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
    channel = connection.channel()
    channel.exchange_declare(exchange=EXCHANGE_NAME, exchange_type='topic', durable=True)

    if args.proof_key:
        city, zone, request_type = args.proof_key.split('.')
        publish(channel, args.proof_key, make_request(city, zone, request_type), 'PROOF ')

    message_number = 0
    try:
        while True:
            message_number += 1
            city = random.choice(tuple(LOCATIONS))
            zone = random.choice(LOCATIONS[city])
            request_type = random.choice(REQUEST_TYPES)
            routing_key = f'{city}.{zone}.{request_type}'
            request = make_request(city, zone, request_type)
            bad = args.bad_every > 0 and message_number % args.bad_every == 0
            if bad:
                del request['zone']
            publish(channel, routing_key, request, 'BAD ' if bad else '')
            time.sleep(args.interval)
    except KeyboardInterrupt:
        pass
    finally:
        connection.close()


if __name__ == '__main__':
    main()


Writing stream/dispatcher.py


## Create `stream/driver.py`

In [4]:
%%writefile stream/driver.py
import argparse
import json

import pika

EXCHANGE_NAME = 'rapido-dispatch'


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('name')
    parser.add_argument('pattern')
    args = parser.parse_args()

    connection = pika.BlockingConnection(pika.ConnectionParameters('localhost'))
    channel = connection.channel()
    channel.exchange_declare(exchange=EXCHANGE_NAME, exchange_type='topic', durable=True)
    queue = channel.queue_declare(queue='', exclusive=True, auto_delete=True).method.queue
    channel.queue_bind(exchange=EXCHANGE_NAME, queue=queue, routing_key=args.pattern)
    channel.basic_qos(prefetch_count=1)

    def handle_request(channel, method, _properties, body):
        try:
            request = json.loads(body)
        except json.JSONDecodeError:
            print(f'{args.name}: invalid JSON; NACKing request')
            channel.basic_nack(method.delivery_tag, requeue=False)
            return

        if not request.get('zone'):
            print(f'{args.name}: missing zone; NACKing request {request}')
            channel.basic_nack(method.delivery_tag, requeue=False)
            return

        print(f'{args.name}: received {method.routing_key}: {request}')
        channel.basic_ack(method.delivery_tag)

    channel.basic_consume(queue=queue, on_message_callback=handle_request, auto_ack=False)
    print(f'{args.name} listening on {queue} with pattern {args.pattern}')
    try:
        channel.start_consuming()
    except KeyboardInterrupt:
        pass
    finally:
        connection.close()


if __name__ == '__main__':
    main()


Writing stream/driver.py


In [5]:
!python -m py_compile stream/dispatcher.py stream/driver.py
!git diff -- stream/dispatcher.py stream/driver.py

## Run the two-minute demonstration

This starts Ravi, Meena, and Control. `--proof-key` sends one predictable valid request for the D2 screenshot, after which the dispatcher continues sending random messages every two seconds.

In [6]:
%%bash
rm -f dispatcher.log ravi.log meena.log control.log
python -u stream/driver.py Ravi 'bangalore.koramangala.*' > ravi.log 2>&1 &
RAVI_PID=$!
python -u stream/driver.py Meena 'bangalore.*.food' > meena.log 2>&1 &
MEENA_PID=$!
python -u stream/driver.py Control 'bangalore.#' > control.log 2>&1 &
CONTROL_PID=$!
sleep 2
timeout 120s python -u stream/dispatcher.py --proof-key bangalore.koramangala.food > dispatcher.log 2>&1 || true
kill $RAVI_PID $MEENA_PID $CONTROL_PID 2>/dev/null || true




 It proves the known routing key was published and reached the three matching driver queues.

In [11]:
!grep -n 'bangalore.koramangala.food' dispatcher.log ravi.log meena.log control.log

dispatcher.log:1:PROOF published bangalore.koramangala.food: {'id': 'e9ecbd2e-941e-4ecf-b93c-28b4df8e04a5', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T18:36:43.464825+00:00'}
dispatcher.log:4:published bangalore.koramangala.food: {'id': '1c837459-d1c2-43d1-b58b-0f0d37727e2f', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T18:36:47.466110+00:00'}
dispatcher.log:36:published bangalore.koramangala.food: {'id': '6415cd09-468b-401d-8852-22f6a7cea1e0', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T18:37:51.484557+00:00'}
dispatcher.log:57:published bangalore.koramangala.food: {'id': '56951407-d6f7-42f5-9eef-cd26fc373182', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T18:38:33.496492+00:00'}
ravi.log:2:Ravi: received bangalore.koramangala.food: {'id': 'e9ecbd2e-941e-4ecf-b93c-28b4df8e04a5', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-



Capture output showing a `BAD published` message and a driver rejecting it because its `zone` field is missing. If no matching bad Bangalore request appears, run the two-minute demonstration again.

In [10]:
!grep -nE 'BAD published|missing zone|NACKing' dispatcher.log ravi.log meena.log control.log

dispatcher.log:11:BAD published chennai.whitefield.ride: {'id': 'e13c3b69-336d-458e-aecd-34176d30ce44', 'city': 'chennai', 'type': 'ride', 'ts': '2026-11-13T18:37:01.469886+00:00'}
dispatcher.log:21:BAD published chennai.tnagar.ride: {'id': 'aaefcccd-6100-4c33-a52b-19a84b53ddf9', 'city': 'chennai', 'type': 'ride', 'ts': '2026-11-13T18:37:21.475871+00:00'}
dispatcher.log:31:BAD published chennai.whitefield.ride: {'id': '0d17d918-7ea5-465c-98f9-ffca1053e704', 'city': 'chennai', 'type': 'ride', 'ts': '2026-11-13T18:37:41.481582+00:00'}
dispatcher.log:41:BAD published chennai.tnagar.food: {'id': '226f818c-0d4e-442e-b818-9d0b83f56bee', 'city': 'chennai', 'type': 'food', 'ts': '2026-11-13T18:38:01.487361+00:00'}
dispatcher.log:51:BAD published chennai.tnagar.ride: {'id': '4e3fe627-9877-4a5f-b1fe-bfdee3746aea', 'city': 'chennai', 'type': 'ride', 'ts': '2026-11-13T18:38:21.493344+00:00'}
dispatcher.log:61:BAD published bangalore.indiranagar.ride: {'id': '087585be-fd22-4c17-8fca-88b7c664938a', 

In [9]:
!head -3 ravi.log
!head -3 meena.log
!head -3 control.log

Ravi listening on amq.gen-fylisuQbol7NqBZcLO6Nug with pattern bangalore.koramangala.*
Ravi: received bangalore.koramangala.food: {'id': 'e9ecbd2e-941e-4ecf-b93c-28b4df8e04a5', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T18:36:43.464825+00:00'}
Ravi: received bangalore.koramangala.food: {'id': '1c837459-d1c2-43d1-b58b-0f0d37727e2f', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T18:36:47.466110+00:00'}
Meena listening on amq.gen-r7P6URynxcQ9dtiOH5qGRw with pattern bangalore.*.food
Meena: received bangalore.koramangala.food: {'id': 'e9ecbd2e-941e-4ecf-b93c-28b4df8e04a5', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T18:36:43.464825+00:00'}
Meena: received bangalore.koramangala.food: {'id': '1c837459-d1c2-43d1-b58b-0f0d37727e2f', 'city': 'bangalore', 'zone': 'koramangala', 'type': 'food', 'ts': '2026-11-13T18:36:47.466110+00:00'}
Control listening on amq.gen-R1vE286V_odipWp25PrcGw with pattern